# Lab Tasks - Solutions

This lab focuses on **community finding** in networks. For the tasks, we will work with a Twitter/X **follower network**. This is a directed network which consists of all follower-followee relationships between a set of users.

Specifically, we will use a dataset collected from the Twitter/X platform, relating to the official user accounts for 30 schools across three Dublin universities. 

The comma-separated file *followers.csv* contains an edge list, where each pair *(A,B)* in the list indicates that user *A* follows user *B*.

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', '{:,.3f}'.format)

### Task 1

Load the follower information from the comma-separated file *followers.csv*.

From this data, create a directed **follower network**.

In [ ]:
g = nx.read_edgelist("followers.csv" , delimiter = ",", create_using=nx.DiGraph)
print(f"Network has {g.number_of_nodes()} nodes and {g.number_of_edges()} edges")

### Task 2

Using the network constructed in Task 1, identify the top 10 most influential users in the network based on: (1) in-degree, (2) betweenness centrality.

In [ ]:
in_deg = pd.Series(dict(g.in_degree()), name="in-degree")
pd.DataFrame(in_deg).sort_values(by="in-degree", ascending=False).head(10)

In [ ]:
between = pd.Series(dict(nx.betweenness_centrality(g)), name="betweenness")
pd.DataFrame(between).sort_values(by="betweenness", ascending=False).head(10)

### Task 3

Using the original network from Task 1, construct an **undirected** version of the follower network where an edge *(X,Y)* exists between two users *X* and *Y* if *X* follows *Y* or *Y* follows *X* or both (i.e., the relationships do **not** have to be reciprocal).

In [ ]:
g2 = g.to_undirected(reciprocal=False)
# check the network size
print(f"Number of nodes: {g2.number_of_nodes()}, Number of edges: {g2.number_of_edges()}")

How many components are present in the new network?

In [ ]:
print(f"Number of connected components: {nx.number_connected_components(g2)}")

### Task 4

Apply the Louvain community finding algorithm to the new undirected network.

In [ ]:
# apply the algorithm 
# note we can set a random seed so we get the same results each time
communities_louvain = nx.community.louvain_communities(g2, seed=100)

# get the community sizes (i.e. number of nodes in each community)
for i, c in enumerate(communities_louvain, start=1):
    print(f"Community {i} has {len(c)} nodes")
print()

# display the nodes in each community
for i, c in enumerate(communities_louvain, start=1):
    print(f"Community {i} members:")
    print(c)    
    print()

Produce a visualisation of the network where nodes are coloured based on their community assignment.

Based on the nodes in each community, what does the community structure appear to represent?

In [ ]:
# create a utility function to draw the network with colours defined by community memberships
def draw_communities(g, communities):
    # create a dictionary which maps a node to a community number
    community_map = {}
    for i, community in enumerate(communities):
        for node in community:
            community_map[node] = i
            
    # get an existing colour palette from matplotlib
    pal = plt.get_cmap("Set3")

    # assign colours to the nodes according to the community map
    community_colors = [community_map[node] for node in g.nodes()]

    # draw the network
    plt.figure(figsize=(10,8))
    plt.margins(0.05, 0.05)
    pos = nx.spring_layout(g, seed=55)
    nx.draw(g, pos, 
            with_labels=True, 
            node_color=community_colors, 
            cmap=pal,  
            edge_color="gray", 
            node_size=600, 
            font_size=8)
    plt.axis("off")
    plt.show()

# apply the new function to produce a visualisation of the communities
draw_communities(g2, communities_louvain)